(ch:classification)=
# 분류

:::{note} 감사의 글

오렐리앙 제롱<font size='2'>Aurélien Géron</font>의 [Hands-On Machine Learning with Scikit-Learn and PyTorch (O'Reilly, 2025)](https://github.com/ageron/handson-mlp)에 사용된 코드를 참고한 강의노트입니다. 보다 심화된 이해를 위해 책 원본을 읽을 것을 강력하게 추천합니다 자료를 공개한 오렐리앙 제롱과 일부 그림 자료를 제공해 준 한빛아카데미에게 진심어린 감사를 전합니다.
:::

:::{seealso} 코드 실행
[(코드 워크아웃) 분류](https://colab.research.google.com/github/codingalzi/code-workout-ml/blob/master/notebooks/code-classification.ipynb)를 병행하여 읽을 것을 권장한다.
:::

MNIST 손글씨 데이터셋을 이용하여 이진 분류와 다중 클래스 분류 모델을 훈련하고, 혼동 행렬과 정밀도·재현율을 통한 성능 평가 및 오차 분석 방법을 다룹니다.

## MNIST 데이터셋

미국의 고등학생과 인구조사국 직원들이 손으로 쓴 70,000개의 숫자 사진으로 구성된 데이터셋이다.

### 데이터 기초 정보

**입력 데이터셋과 타깃 데이터셋**

입력 데이터 샘플은 손글씨 사진을 `28x28=784` 크기의 1차원 어레이로 변환된 값으로 제공되며,
라벨(타깃)은 정수가 아니라 `'0'`, `'1'`, ..., `'9'`처럼 문자열로 지정되어 있다.

| 구분 | 변수명 | 설명 |
| :--- | :--- | :--- |
| **입력 데이터셋** | `X` | `28x28=784` 크기의 1차원 어레이로 제공되는 손글씨 사진 데이터 샘플 7만개로 구성된 넘파이 2차원 어레이 (모양: `(70000, 784)`) |
| **타깃 데이터셋** | `y` | 각 손글씨 사진에 해당하는 정답 라벨이 `'0'`부터 `'9'`까지의 문자열로 지정된 넘파이 1차원 어레이 (모양: `(70000, )`) |

:::{note} 사진 픽셀 데이터

모델 훈련에 사용되는 입력 데이터셋의 모든 샘플은 길이가 784인 1차원 어레이로 주어졌다.
어레이의 각 항목은 `28x28` 모양의 손글씨 사진에 포함된 하나의 픽셀(화소)값에 해당한다.
즉, 사진에 포함된 모든 픽셀이 입력 데이터 샘플의 특성이 된다.
:::

첫 입력 데이터 샘플에 담긴 손글씨 사진은 다음과 같다. 숫자 5를 쓴 것으로 보인다. 실제 타깃도 숫자 5이다.

<div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/mnist_digit_5.jpg?raw=true" width="300"/></div>

첫 100 개의 손글씨 사진은 다음과 같다.

<div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/homl03-01.png?raw=true" width="450"/></div>

### 머신러닝 훈련 모델 선택

손글씨 사진이 표현하는 0부터 9까지의 정수를 예측하는 모델을 훈련시키려 한다.

* 지도학습: 각 사진이 담고 있는 숫자가 타깃(라벨)으로 지정되어 있음.
* 모델: 사진 데이터를 분석하여 0부터 9까지, 총 10개의 범주로
    분류 해야함.
    이를 위해 **다중 클래스 분류**<font size='2'>multiclass classification</font>를
    지원하는 모델 활용.
* 실시간 훈련 여부: 확률적 경사하강법 분류기와 랜덤 포레스트 분류기를 활용한 배치 학습 실행

### 훈련셋과 테스트셋

입력 데이터셋 `X`와 `y`는 이미 6:1 의 비율로 훈련셋과 테스트셋으로 분류되어 있다.
모든 샘플은 10개의 숫자에 대해 균등하게 무작위로 잘 섞여 있다.

| 구분 | 변수명 | 설명 |
| :--- | :--- | :--- |
| 훈련 입력 데이터셋 | `X_train` | 앞쪽 60,000개 사진 데이터로 구성된 `(60000, 784)` 모양의 넘파이 2차원 어레이 |
| 훈련 타깃 데이터셋 | `y_train` | 6만개의 라벨로 구성된 `(60000, )` 모양의 넘파이 1차원 어레이 |
| 테스트 입력 데이터셋 | `X_test` | 나머지 1만개의 사진 데이터로 구성된 `(10000, 784)` 모양의 넘파이 2차원 어레이 |
| 테스트 타깃 데이터셋 | `y_test` | 1만개의 라벨로 구성된 `(10000, )` 모양의 넘파이 1차원 어레이 |

## 이진 분류기: 숫자-5 감별기

10개의 클래스로 분류하는 다중 클래스 모델을 훈련하기 전에 먼저
손글씨 사진이 숫자 5를 표현하는지 여부를 판단하는 
**이진 분류기**<font size='2'>binary classifier</font>를 훈련시킨다.
이를 통해 분류기의 기본 훈련 과정과 성능 평가 방법을 알아본다.

숫자-5 감별기를 이진 분류기 훈련을 활용하여 생성하기 위해 
타깃 데이터셋(`y_train_5`)을 새로 설정한다.

| `y_train_5`의 라벨 | 설명 |
| :---: | :--- |
| `1` | 숫자 5를 가리키는 사진의 라벨 |
| `0` | 숫자 5 이외의 수를 가리키는 사진의 라벨 |

먼저 `SGDClassifier` 클래스를 이용하여 숫자-5 감별기를 훈련시킨다.
`SGDClassifier` 분류기는 
**확률적 경사하강법**<font size='2'>stochastic gradient descent</font> 분류기라고도 불린다.
훈련속도가 빨라 대용량 데이터셋을 활용한 훈련에 활용되는 선형회귀를 활용한 분류기이며, 
훈련방식은 다음 장에서 자세히 다룬다.

```python
sgd_clf = SGDClassifier(random_state=42)
sgd_clf.fit(X_train, y_train_5)
```

`sgc_clf` 모델은 784 개의 픽셀 정보를 이용하여 해당 사진 샘플이 5를 표현하는지 여부를 판정하도록 훈련된다.

## 이진 분류기 성능 측정

이진 분류기를 포함해서 훈련이 완료된 모든 분류기의 성능 평가는 일반적인 보통 다음 세 가지를
활용하여 진행한다.

* 정확도
* 정밀도와 재현율
* ROC 곡선의 AUC

여기서는 정확도, 정밀도, 재현율을 활용하여
먼저 이진 분류기의 성능을 평가하는 과정을 상세하게 소개한다.
ROC 곡선의 AUC에 대해서는 [분류: ROC 및 AUC](https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc?hl=ko)를 참고할 것을 추천한다.

분류기의 성능 평가 세 가지 방식 모두 혼동 행렬을 활용한다.

### 혼동 행렬

**혼동 행렬**<font size="2">confusion matrix</font>은 클래스별 예측 결과를 정리한 행렬이다.
훈련이 완료된 이진 분류기인 숫자-5 감별기에 대한 혼동 행렬은 아래와 같은 (2, 2) 모양의 2차원 (넘파이) 어레이로 생성된다.

```python
array([[53892,   687],
       [ 1891,  3530]])
```

아래 그림은 혼동 행렬에 포함된 각 정수의 의미를 설명하기 위해
숫자-5 감별기의 판정 결과를 단순화해서 보여준다. 

<p><div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/homl03-02.png?raw=true" width="500"/></div></p>

아래 표는 위 그림에 포함된 용어와 각 칸에 포함된 손글씨 이미지들에 대한 설명을 담고 있다.

| 분류 | 예측: 음성 (5 아니라고 판정) | 예측: 양성 (5라고 판정) |
| :--- | :--- | :--- |
| 실제: 음성<br>(5가 아님) | TN (참 음성, True Negative)<br>5가 아닌 숫자를 5가 아니라고 정확히 예측<br>(예: 8, 3, 9, 7, 2) | FP (거짓 양성, False Positive)<br>5가 아닌 숫자를 5라고 잘못 예측<br>(예: 6) |
| 실제: 양성<br>(5임) | FN (거짓 음성, False Negative)<br>실제 5인 숫자를 5가 아니라고 잘못 예측 | TP (참 양성, True Positive)<br>실제 5인 숫자를 5라고 정확히 예측 |

위 그림을 토대로한 혼동 행렬은 다음과 같이 작성된다.

```python
array([[5, 1],
       [2, 3]])
```

### 정확도

**정확도**<font size='2'>accuracy</font>는 라벨을 정확하기 맞힌 비율이다.
훈련된 SGD 분류기의 정확도는 다음과 같이 96% 정도로 매우 높게 계산된다.

```
accuracy = (TP + TN)/(TP+FP+TN+FN)
         = (3530 + 53892)/(3530 + 687 + 53892 + 1891)
         = 0.957
```

**정확도의 한계**

정확도가 96% 정도로 매우 좋은 결과로 보인다. 하지만 "무조건 5가 아니다" 라고 예측하는 모델도 90%의 정확도를 보인다. 이유는 6만개의 훈련셋에 0부터 9까지의 손글씨 사진이 균등하게 포함되어 있어서, 5가 아닌 다른 숫자를 표현하는 손글씨가 전체의 90% 정도를 차지하기 때문이다.

특정 범주에 속하는 데이터가 상대적으로 너무 많을 경우 정확도는 신뢰하기 어렵다.
이런 경우엔 정확도 보다는 정밀도와 재현율을 이용한 평가 방식이 권장된다.

### 정밀도와 재현율

**정밀도**<font size="2">precision</font>는 양성 예측의 정확도를 가리킨다.
여기서는 숫자 5라고 예측된 값들 중에서 진짜로 5인 숫자들의 비율이다. 

```
precision = TP / (TP + FP) = 3530 / (3530 + 687) = 0.837
```

**재현율**<font size='2'>recall</font>은 양성 샘플에 대한 정확도, 즉, 분류기가 정확하게 감지한 양성 샘플의 비율이며,
**참 양성 비율**<font size="2">true positive rate</font>로도 불린다.

```
recall = TP / (TP + FN) = 3530 / (3530 + 1891) = 0.651
```

**정밀도와 재현율의 상대적 중요도**

모델 사용의 목적에 따라 정밀도와 재현율의 중요도가 다를 수 있다.

* 재현율이 보다 중요한 경우: 암진단처럼 예측이 틀렸을 경우 큰 책임이 따르는 경우. 
    * 정밀도: 암이 있다고 진단된 경우 중에 실제로도 암이 있는 경우의 비율
    * 재현율: 암으로 판정해야 하는 경우 중에서 양성 암진단으로 결론내린 경우의 비율
    * 다른 예제: 금융 사기 여부, 태풍 예보

* 정밀도가 보다 중요한 경우: 아이에게 보여줄 안전한 동영상 선택처럼 일부 양성 모델을 놓치더라도 음성이 양성으로 판정되는 경우가 적어야 하는 경우.
    * 정밀도: 안전하다고 판단된 동영상 중에서 실제로도 안전한 동영상의 비율
    * 재현율: 실제로 좋은 동영상 중에서 좋은 동영상이라고 판정되는 동영상 비율
    * 다른 예제: 스팸 필터링

### 정밀도와 재현율의 상충관계

분류기의 **결정 함수**<font size="2">decision function</font>는 각 샘플에 대해 점수를 계산하며
이 점수가 **결정 임계값**<font size="2">decision threshold</font>보다
같거나 크면 양성, 아니면 음성으로 판단한다.
예를 들어 `SGDClassifier`는 `decision_function()` 메서드를 결정 함수로 이용하며,
결정 함숫값이 0보다 작으면 음성, 0보다 같거나 크면 양성으로 판정한다.

정밀도와 재현율은 상호 반비례 관계이다.
즉, 한쪽이 증가하면 다른쪽이 감소하는 상충 관계이다.
따라서 정밀도와 재현율 사이의 적절한 비율을 유지하는 분류기를 찾아야 한다. 
정밀도와 재현율의 비율은 모델이 사용하는 **결정 임곗값**에 따라 달라진다.

아래 예제가 정밀도와 재현율의 트레이드오프 관계를 잘 보여준다.

- 결정 임곗값의 위치에 따라 정밀도와 재현율이 서로 다른 방향으로 움직인다.
- 결정 임곗값이 클 수록 분류기의 정밀도는 올라가지만 재현율은 떨어진다.
- 결정 임곗값이 작을 수록 분류기의 정밀도는 내려가지만 재현율은 올라간다.

:::{prf:example} 정밀도와 재현율의 트레이드오프
:label: exp_decision_threshold

아래 그림에서 세 개의 화살표 (a), (b), (c)는 서로 다른 결정 임곗값을 가리키며, 
화살표 윗쪽에 위치한 정밀도와 재현율은 해당 결정 임곗값을 기준으로
주어진 샘플의 양성, 음성 여부를 판단할 경우의 정밀도와 재현율이다. 

<div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/homl03-03.png?raw=true" width="500"/></div>

- 경우 (a)
    - 정밀도 80%: 양성으로 예측된 5개의 샘플 중 정말로 5를 가리키는 샘플 4개, 아닌 샘플 1개
    - 재현율 67%: 실제로 5인 샘플 총 6개 중에 5라고 판정된 샘플 4개
- 경우 (b)
    - 정밀도 75%: 양성으로 예측된 8개의 샘플 중 정말로 5를 가리키는 샘플 6개, 아닌 샘플 2개
    - 재현율 100%: 실제로 5인 샘플 총 6개 중에 5라고 판정된 샘플 6개
- 경우 (c)
    - 정밀도 100%: 양성으로 예측된 3개의 샘플 중 정말로 5를 가리키는 샘플 3개, 아닌 샘플 0개
    - 재현율 50%: 실제로 5인 샘플 총 6개 중에 5라고 판정된 샘플 3개
:::

**결정 함수와 결정 임곗값**

결정 함수<font size='2'>decision function</font>는 
분류 모델이 주어진 데이터 샘플의 클래스를 판별하기 위해 내부적으로 
각 샘플에 대해 점수를 계산할 때 사용하는 메서드(함수)다. 
양성 예측의 기준이 되는 **결정 임곗값**<font size='2'>decision threshold</font>과
이 점수를 비교하여, 결정 함숫값이 임곗값보다 같거나 크면 양성, 작으면 음성으로 판정한다.

결정 임곗값을 높이면 양성으로 판정받기 어려워져 정밀도는 올라가고 재현율은 떨어집니다. 반대로 임곗값을 낮추면 양성 판정이 쉬워져 정밀도는 떨어지고 재현율은 올라가게 됩니다.

예를 들어 처음 10개 샘플에 대한 결정 함숫값은 다음과 같다.
첫째 샘플의 결정 함숫값만 양수이고 나머지 9개는 음수다.
따라서 첫째 샘플만 5로 판정되고 나머자 9개는 5가 아니다라고 판정된다.

```
array([1200.93051237,
       -26883.79202424,
       -33072.03475406,
       -15919.5480689,
       -20003.53970191,
       -16652.87731528,
       -14276.86944263,
       -23328.13728948,
       -5172.79611432,
       -13873.5025381  ])
```

**결정 임곗값/정밀도/재현율 그래프**

아래 그래프는 `SGDClassifier` 모델을 숫자-5 감별기로 훈련시킨 결과를 이용한다.
그래프는 x-축의 결정 임곗값에 따른 정밀도와 재현율의 변화를 보여준다.
결정 임곗값이 커질 때 정밀도가 순간적으로 떨어질 수 있지만 결국엔 계속해서 상승한다.

<p><div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/homl03-04.png?raw=true" width="500"/></div></p>

`SGDClassifier` 는 0을 결정 임곗값으로 사용하기에 정밀도는 84% 정도, 재현율은 65% 정도다.
반면에 검정 수직 점선은 정밀도는 90%, 재현율은 50% 정도가 되게 하는 결정 임곗값을 보여준다.
결정 임곗값을 변경하여 원하는 정밀도와 재현율을 갖는 숫자-5 감별기를 구현하려면 
수동으로 분류기를 구현해야 한다 ([90% 정밀도 분류기 구현](https://colab.research.google.com/github/codingalzi/handson-ml3/blob/master/notebooks/code_classification.ipynb#scrollTo=cKfpQLyuCHkf) 참고).

**정밀도/재현율 그래프**

위 그래프를 재현율 대 정밀도 그래프로 변환하면 다음과 같다.
결정 임곗값(threshold)을 낮춰 재현율을 올리면 정밀도는 떨어지는 상호 상충관계를 잘 보여준다.

<div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/homl03-05.png?raw=true" width="400"/></div>

## 다중 클래스 분류

**다중 클래스 분류**<font size="2">multiclass classification</font>는 
세 개 이상의 범주(클래스)로 샘플을 분류한다.
예를 들어, MNIST 손글씨 숫자 문제는 입력값이 주어지면 
0부터 9까지 10개의 범주로 분류하는 다중 클래스 모델을 훈련시키는 일이다.

**다중 클래스 분류 지원 모델**

아래 모델은 이진 분류와 다중 클래스 분류를 지원한다.

* `LogisticRegression` 모델
* `RandomForestClassifier` 모델
* `SGDClassifier` 모델
* `SVC` 모델

이중에 `LogisticRegression` 모델은 딥러닝 모델에서도 많이 활용된다.

- 이진 분류: 로지스틱 회귀 활용
- 소프트맥스 회귀: 다중 클래스 분류

자세한 내용은 {numref}`%s장 <ch:trainingModels>`에서 다룬다.

**다중 클래스 분류 모델 교차 검증**

숫자-5 감별기의 경우와는 달리 
0부터 9까지의 범주로 분류하는 다중 클래스 분류는
각 숫자에 해당하는 데이터가 고르게 분포되어 있어서 
정확도를 기준으로 교차 검증을 진행해도 괜찮다.

아래 코드를 이용하여 `SGDClassifier` 모델에 대해
교차 검증으로 정확도를 계산하면 86.7% 정도로 확인된다.
참고로 정밀도와 재현율을 교차 검증으로 확인하려면 
`scoring` 키워드의 인자로 `"precision"` 또는 `"recall"`을 지정한다.

```python
from sklearn.model_selection import cross_val_score

cross_val_score(sgd_clf, X_train, y_train, cv=3, scoring="accuracy")
```

**스케일링의 중요성**

하지만 표준화 스케일링만 해도 모델의 예측 정확도가 89.7% 까지 향상된다.

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.astype("float64"))
cross_val_score(sgd_clf, X_train_scaled, y_train, cv=3, scoring="accuracy")
```

그런데 `MinMaxScaling` 전처리를 적용하면 성능이 91.05까지 좋아진다.
손글씨 사진이 아무래도 픽셀 정보를 담고 있는 반면에 표준화 스케일링을 진행하면
아무래도 음수로 변환되는 픽셀도 있다.
따라서 모델의 예측 성능에 영향을 미치는 것으로 추정되지만 정확한 이유는 불명확하다.

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train.astype("float64"))
cross_val_score(sgd_clf, X_train_scaled, y_train, cv=3, scoring="accuracy")
```

## 다중 클래스 분류기 모델의 오류 분석

그리드 탐색, 랜덤 탐색 등을 이용한 모델 튜닝 과정을 실행하여 최선의 모델을 찾았다고 가정한다.
이제 오류 분석을 통해 모델의 성능을 평가하고 개선시키는 방안을 모색하는 과정을 살펴 본다.
먼저 훈련된 모델의 성능을 평가하기 위해 혼동 행렬을 활용한다.

### 다중 클래스 분류 모델의 혼동 행렬

먼저 `cross_val_predict()` 함수를 이용하여 교차 검증 방식으로 표준화 스케일링된 훈련셋에 대한 모델의 예측값을 계산한다.

```python
from sklearn.model_selection import cross_val_predict

y_train_pred = cross_val_predict(sgd_clf, X_train_scaled, y_train, cv=3)
```

아래 왼쪽 사진은 훈련된 분류 모델의 혼동 행렬을 색상을 이용하여 표현한다.
대각선 상에 위치한 색상이 밝은 것은 분류가 대체로 잘 이루어졋음을 의미한다. 
다만 5번 행이 상대적으로 어두운데 이는 숫자 5의 분류 정확도가 상대적으로 낮기 때문이다.

```python
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_train, y_train_pred)
```

반면에 아래 오른쪽 사진은 숫자별 비율로 변환하였다. 
즉, 행별로 비율의 합이 100%가 되도록 정규화하였다.

```python
ConfusionMatrixDisplay.from_predictions(y_train, y_train_pred,,
                                        normalize="true", 
                                        values_format=".0%")
```

<p><div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/homl03-08.png?raw=true" width="100%"/></div></p>

**오차율 활용**

위 오른쪽 사진에서 8로 오인된 사진이 많았음을 알 수 있다.
실제로 올바르게 예측된 샘플을 제외한 다음에 행별로 오인된 숫자의 비율을 확인하면
아래 왼쪽 사진과 같다.
9번 칸과 8번 칸이 상대적으로 많이 밝으며, 
이는 많은 숫자가 9 또는 8로 오해되었다는 의미다. 

```python
sample_weight = (y_train_pred != y_train)

ConfusionMatrixDisplay.from_predictions(y_train, y_train_pred,
                                        sample_weight=sample_weight,
                                        normalize="true", 
                                        values_format=".0%")
```

아래 오른쪽 사진은 열 별로 정규화한 결과를 보여준다. 
예를 들어, 7로 오인된 사진 중에 숫자 9 사진의 비율이 41%임을 알 수 있다.

```python
ConfusionMatrixDisplay.from_predictions(y_train, y_train_pred,
                                        sample_weight=sample_weight,
                                        normalize="pred", 
                                        values_format=".0%")
```

<p><div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/homl03-09.png?raw=true" width="100%"/></div></p>

**개별 오류 확인**

위 오른쪽 사진에 의하면 5로 오인된 사진 중에서 숫자 3 사진의 비율이
34%로 가장 높다.
실제로 혼동 행렬과 유사한 행렬을 3과 5에 대해 나타내면 다음과 같다.

* 음성: 3으로 판정
* 양성: 5로 판정

<div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/homl03-10.png?raw=true" width="500"/></div>

### 데이터 증식

사람 눈으로 보더라도 3과 5의 구분이 매우 어려울 수 있다.
여기서 사용한 SGD 분류 모델은 선형 회귀를 사용하기에 특히나 성능이 좋지 않다.
따라서 보다 좋은 성능의 모델을 사용할 수도 있지만
기본적으로 보다 많은 훈련 사진이 필요하다.
새로운 사진을 구할 수 있으면 좋겠지만 일반적으로 매우 어렵다.
반면에 기존의 사진을 조금씩 회전하거나, 뒤집거나, 이동하는 방식 등으로
보다 많은 사진을 훈련셋에 포함시킬 수 있다.
이런 방식을 데이터 증식<font size="2">data augmentation</font>이라 부른다.
데이터 증식에 대한 보다 상세한 설명은
[데이터 증식 연습문제](https://colab.research.google.com/github/codingalzi/handson-ml3/blob/master/notebooks/code_classification.ipynb#scrollTo=7GQbYjmuCHkn)를 참고한다.

## 연습문제

참고: [(실습) 분류 1](https://colab.research.google.com/github/codingalzi/handson-ml3/blob/master/practices/practice_classification_1.ipynb) 과
[(실습) 분류 2](https://colab.research.google.com/github/codingalzi/handson-ml3/blob/master/practices/practice_classification_2.ipynb)